# Weather Agent — Bug Report & Fix

This notebook documents the **8 bugs** found in the broken weather agent codebase, explains each fix, and demonstrates the working agent.

---

## Bug Summary

| # | File | Line | Bug | Fix |
|---|------|------|-----|-----|
| 1 | `main.py` | 32 | `"_main_"` (missing underscores) | `"__main__"` |
| 2 | `graph.py` | 19–20 | `fetch_weather_data` node bypassed (edge skips it) | Added proper edge chain |
| 3 | `graph.py` | 23 | Graph never compiled (`weather_agent = builder`) | `builder.compile()` |
| 4 | `nodes.py` | 38 | `state["location_data"] = {}` discards API response | `= location_data` |
| 5 | `nodes.py` | 132 | `wind_unit` variable commented out | Uncommented |
| 6 | `nodes.py` | 154 | `wind_unit` used as string literal in f-string | `{wind_unit}` |
| 7 | `config.py` | 53–55 | Duplicate `WEATHER_API_BASE_URL: str = ""` overrides real URL; `TEMP_MIN = str = "Needs Debugging"` malformed | Removed duplicate; `TEMP_MIN: float = 0.0` |
| 8 | `helper_functions.py` | 14 | `if temp_celsius:` — truthy check always returns garbage string | `if temp_celsius < config.TEMP_MIN:` → `"freezing"` |

---
## Bug 1 — `main.py`: `_main_` entry-point typo

**Original:**
```python
if __name__ == "_main_":   # only ONE underscore each side
    main()
```
The guard never fires when the script is run directly, so `main()` is never called.

**Fix:**
```python
if __name__ == "__main__":  # two underscores each side
    main()
```

---
## Bug 2 — `graph.py`: `fetch_weather_data` node never executed

**Original edges:**
```python
builder.add_edge(START, "fetch_location_data")
builder.add_edge("fetch_location_data", "generate_weather_info")  # skips weather fetch!
builder.add_edge("generate_weather_info", END)
```
The `fetch_weather_data` node is registered but never wired into the graph — it is dead code. `generate_weather_info` then fails because `weather_data` is `None`.

**Fix:**
```python
builder.add_edge(START, "fetch_location_data")
builder.add_edge("fetch_location_data", "fetch_weather_data")
builder.add_edge("fetch_weather_data", "generate_weather_info")
builder.add_edge("generate_weather_info", END)
```

---
## Bug 3 — `graph.py`: Graph never compiled

**Original:**
```python
weather_agent = builder   # just the builder object, not a runnable graph
```
Calling `.invoke()` on a `StateGraph` builder (not a compiled graph) raises an `AttributeError`.

**Fix:**
```python
weather_agent = builder.compile()
```

---
## Bug 4 — `nodes.py`: Location data thrown away

**Original:**
```python
location_data = response.json()   # ← correct data in local var
# ... validation ...
state["location_data"] = {}       # ← overwrites with empty dict!
```
After a successful API call and validation, the actual data is discarded and replaced with an empty dict. Every downstream node then fails with `KeyError`.

**Fix:**
```python
state["location_data"] = location_data
```

---
## Bug 5 & 6 — `nodes.py`: `wind_unit` commented out then used as string literal

**Original:**
```python
# wind_unit = units.get("windspeed", "km/h")   # commented out
...
f"• Wind: {windspeed} wind_unit"   # 'wind_unit' is a literal string!
```
Output would show: `• Wind: 12.5 wind_unit`

**Fix:**
```python
wind_unit = units.get("windspeed", "km/h")   # uncommented
...
f"• Wind: {windspeed} {wind_unit}"           # variable in braces
```

---
## Bug 7 — `config.py`: Duplicate field + malformed assignment

**Original (end of Config class):**
```python
    WEATHER_API_BASE_URL: str = ""      # duplicate — overrides the correct URL above!
    TEMP_MIN = str = "Needs Debugging"  # invalid Python; assigns str type to TEMP_MIN
```
The duplicate `WEATHER_API_BASE_URL` blanks out the real URL so every weather API call hits an empty string and fails. `TEMP_MIN = str = "Needs Debugging"` is syntactically odd — it sets `TEMP_MIN` to the `str` type and `str` to `"Needs Debugging"`, not what was intended.

**Fix:**
```python
    TEMP_MIN: float = 0.0   # remove duplicate; fix TEMP_MIN type annotation
```

---
## Bug 8 — `helper_functions.py`: Broken temperature classification

**Original:**
```python
def classify_temperature(temp_celsius: float) -> str:
    if temp_celsius:          # truthy — True for ANY non-zero temperature
        return config.TEMP_MIN  # returned "Needs Debugging" for everything
    elif temp_celsius < config.TEMP_COLD:
        ...
```
Because `if temp_celsius:` is truthy for every non-zero number, every temperature classification returned `"Needs Debugging"` (the bad string from Bug 7). Only a temperature of exactly `0` would fall through.

**Fix:**
```python
    if temp_celsius < config.TEMP_MIN:   # below 0°C → freezing
        return "freezing"
    elif temp_celsius < config.TEMP_COLD:
        ...
```

---
## Running the Fixed Agent

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), 'weather_agent'))

from weather_agent.graph import weather_agent
from weather_agent.components.state import WeatherAgentState

state = WeatherAgentState(
    name="Maith",
    location_data=None,
    weather_data=None,
    weather_info=None
)

final_state = weather_agent.invoke(state)

print("=" * 60)
print("WEATHER INFORMATION")
print("=" * 60)
print(final_state["weather_info"])

WEATHER INFORMATION
Time: 13:30 UTC | 19:00 (UTC+0530)

Good evening, Maith!

Your current location: Nagpur, Maharashtra, India

Current weather conditions:
• Partly cloudy
• Temperature: 38.3°C (hot)
• Wind: 8.5 km/h


---
## Unit Tests for Each Bug Fix

In [2]:
# Bug 1 — __main__ guard
assert '__main__' in open('weather_agent/main.py').read(), 'FAIL: Bug 1'
print('Bug 1 PASS: __main__ guard is correct')

# Bug 3 — graph is compiled
from langgraph.graph.state import CompiledStateGraph
assert isinstance(weather_agent, CompiledStateGraph), 'FAIL: Bug 3'
print('Bug 3 PASS: graph is a compiled CompiledStateGraph')

# Bug 7 — WEATHER_API_BASE_URL is not empty
from weather_agent.components.config import config
assert config.WEATHER_API_BASE_URL != '', 'FAIL: Bug 7'
print(f'Bug 7 PASS: WEATHER_API_BASE_URL = {config.WEATHER_API_BASE_URL}')

# Bug 8 — temperature classification
from weather_agent.components.helper_functions import classify_temperature
assert classify_temperature(-5) == 'freezing',  'FAIL: Bug 8 (freezing)'
assert classify_temperature(5)  == 'cold',      'FAIL: Bug 8 (cold)'
assert classify_temperature(14) == 'cool',      'FAIL: Bug 8 (cool)'
assert classify_temperature(22) == 'comfortable','FAIL: Bug 8 (comfortable)'
assert classify_temperature(28) == 'warm',      'FAIL: Bug 8 (warm)'
assert classify_temperature(35) == 'hot',       'FAIL: Bug 8 (hot)'
print('Bug 8 PASS: all temperature classifications correct')

Bug 1 PASS: __main__ guard is correct
Bug 3 PASS: graph is a compiled CompiledStateGraph
Bug 7 PASS: WEATHER_API_BASE_URL = https://api.open-meteo.com/v1/forecast
Bug 8 PASS: all temperature classifications correct


---
## Summary

| Bug | File | Root Cause | Impact |
|-----|------|------------|--------|
| 1 | `main.py` | Typo in dunder name | Script never ran |
| 2 | `graph.py` | Missing edge to `fetch_weather_data` | Weather data never fetched |
| 3 | `graph.py` | Graph not compiled | `invoke()` crashed |
| 4 | `nodes.py` | Location result discarded | All downstream nodes fail |
| 5 | `nodes.py` | `wind_unit` commented out | `NameError` at runtime |
| 6 | `nodes.py` | `wind_unit` as string literal | Wrong output |
| 7 | `config.py` | Duplicate field + malformed assignment | Blank API URL, crashes |
| 8 | `helper_functions.py` | Truthy check on float | Garbage classification for all temps |